# SQL Joins — Complete Reference

| Pattern | Use Case |
|---------|----------|
| INNER / LEFT / FULL OUTER | Basic join types and NULL behavior |
| Self-join | Compare rows within same table |
| Anti-join | Rows in A but not in B |
| CROSS JOIN | Cartesian product, combination generation |
| Multi-table join order | Minimize intermediate row counts |

**Mental model**: A JOIN combines rows from two tables based on a condition. The type determines what happens when no match exists:

```
INNER JOIN:      only matching rows from both sides
LEFT JOIN:       all left rows; NULLs for unmatched right
RIGHT JOIN:      all right rows; NULLs for unmatched left
FULL OUTER JOIN: all rows from both; NULLs where no match
CROSS JOIN:      every left row × every right row (no ON condition)
```

## Visual Model

```
customers (A)       orders (B)           Results
─────────────       ──────────────       ─────────────────────────────────────
id  name            order_id  cust_id    INNER JOIN (only matching)
1   Alice           1001      1          Alice → 1001
2   Bob             1002      1          Alice → 1002
3   Carol           1003      2          Bob   → 1003
4   Dave                                 (Carol=3, Dave=4 dropped — no orders)

                                         LEFT JOIN (all left, nulls for right)
                                         Alice → 1001
                                         Alice → 1002
                                         Bob   → 1003
                                         Carol → NULL  ← kept
                                         Dave  → NULL  ← kept

ANTI-JOIN pattern (LEFT JOIN + IS NULL)
───────────────────────────────────────
SELECT c.* FROM customers c
LEFT JOIN orders o ON c.id = o.cust_id
WHERE o.cust_id IS NULL;
→ Carol, Dave  (customers with no orders)

SELF-JOIN (manager hierarchy)
───────────────────────────────
SELECT e.name AS employee, m.name AS manager
FROM employees e LEFT JOIN employees m ON e.manager_id = m.id;
→ Alice | NULL (CEO), Bob | Alice, Carol | Alice...
```

## Setup — Libraries and Config

In [ ]:
import sqlite3

print(f"sqlite3 version: {sqlite3.sqlite_version}")

def make_db():
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.executescript("""
        CREATE TABLE customers (
            customer_id INTEGER PRIMARY KEY,
            name TEXT, region TEXT, tier TEXT
        );
        INSERT INTO customers VALUES
            (1,'Alice','East','Gold'),
            (2,'Bob','West','Silver'),
            (3,'Carol','East','Bronze'),
            (4,'Dave','West','Gold'),
            (5,'Eve','North','Silver');  -- no orders

        CREATE TABLE orders (
            order_id INTEGER PRIMARY KEY,
            customer_id INTEGER,
            product_id INTEGER,
            amount REAL,
            order_date TEXT,
            status TEXT
        );
        INSERT INTO orders VALUES
            (101,1,10,250,'2024-01-05','completed'),
            (102,1,20,180,'2024-01-20','completed'),
            (103,2,10,500,'2024-01-08','completed'),
            (104,3,30, 75,'2024-01-12','cancelled'),
            (105,1,10,320,'2024-02-01','completed'),
            (106,2,20,200,'2024-02-14','completed'),
            (107,4,30,1200,'2024-02-20','completed'),
            (108,6,10, 90,'2024-03-01','completed');  -- orphan (customer_id=6 doesn't exist)

        CREATE TABLE products (
            product_id INTEGER PRIMARY KEY,
            name TEXT, category TEXT, price REAL
        );
        INSERT INTO products VALUES
            (10,'Widget','Electronics',250),
            (20,'Gadget','Electronics',180),
            (30,'Drill', 'Tools',      320),
            (40,'Shirt', 'Clothing',    75);  -- no orders for Shirt

        CREATE TABLE employees (
            id INTEGER PRIMARY KEY,
            name TEXT, manager_id INTEGER, department TEXT, salary REAL
        );
        INSERT INTO employees VALUES
            (1,'Alice',NULL,'Exec',200000),
            (2,'Bob',1,'Eng',150000),
            (3,'Carol',1,'Sales',140000),
            (4,'Dave',2,'Eng',120000),
            (5,'Eve',2,'Eng',115000),
            (6,'Frank',3,'Sales',100000);
    """)
    conn.commit()
    return conn

def run(conn, sql, title=""):
    cur = conn.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    col_w = [max(len(c), max((len(str(r[c])) for r in rows), default=4)) for c in cols]
    sep = "  ".join("-" * w for w in col_w)
    header = "  ".join(c.ljust(w) for c, w in zip(cols, col_w))
    if title:
        print(f"\n=== {title} ===")
    print(header)
    print(sep)
    for r in rows:
        print("  ".join(str(r[c]).ljust(w) for c, w in zip(cols, col_w)))

conn = make_db()
print("Database ready.")

## Decision Map — Which Join?

```
Goal?
│
├─ Only rows where both sides match?          → INNER JOIN
├─ All left rows, even without a match?       → LEFT JOIN
├─ All right rows, even without a match?      → RIGHT JOIN (or swap table order + LEFT)
├─ All rows from both, NULLs where no match?  → FULL OUTER JOIN
│                                               (SQLite: UNION of LEFT + RIGHT)
├─ Rows in A that have NO match in B?         → LEFT JOIN ... WHERE B.key IS NULL
│                                               (anti-join / NOT EXISTS)
├─ Compare row to other rows in SAME table?   → SELF JOIN
└─ All combinations (no condition)?           → CROSS JOIN

NULL BEHAVIOR TRAP:
  INNER JOIN: rows with NULL join key NEVER match
  LEFT JOIN:  NULL join key on left → no match → right cols NULL
  NULL = NULL is FALSE in SQL (use IS NULL, IS NOT NULL)

FANOUT TRAP (unexpected row multiplication):
  One-to-many: 1 customer × 5 orders → 5 rows (expected)
  Many-to-many: orders × tags (multiple tags per order) → fanout!
  Fix: aggregate before joining, or use EXISTS/IN instead of JOIN
```

## Pattern 1 — INNER / LEFT / FULL OUTER JOIN

In [ ]:
# INNER JOIN: only rows with matching keys on both sides
run(conn, """
    SELECT c.name, o.order_id, o.amount, o.status
    FROM customers c
    INNER JOIN orders o ON c.customer_id = o.customer_id
    ORDER BY c.name, o.order_id
""", "INNER JOIN: matched rows only (Eve excluded, orphan order excluded)")

# LEFT JOIN: all customers, NULL for those without orders
run(conn, """
    SELECT c.name, c.tier, o.order_id, o.amount
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    ORDER BY c.name, o.order_id
""", "LEFT JOIN: all customers (Eve appears with NULLs)")

# FULL OUTER JOIN simulation (SQLite has no FULL OUTER)
run(conn, """
    -- All customers + all orders (both sides preserved)
    SELECT c.name AS customer, o.order_id, o.amount
    FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id
    UNION
    SELECT c.name, o.order_id, o.amount
    FROM orders o LEFT JOIN customers c ON o.customer_id = c.customer_id
    ORDER BY customer, order_id
""", "FULL OUTER JOIN simulation: orphan order (NULL customer) + no-order customers")

## Pattern 2 — Self-Join

In [ ]:
# Self-join: join a table to itself using different aliases
# Use cases: org hierarchy (employee/manager), compare rows (e.g., consecutive dates)

# Employee and their manager
run(conn, """
    SELECT
        e.name AS employee,
        e.department,
        e.salary,
        m.name AS manager,
        m.salary AS manager_salary,
        ROUND(100.0 * (m.salary - e.salary) / m.salary, 1) AS gap_pct
    FROM employees e
    LEFT JOIN employees m ON e.manager_id = m.id
    ORDER BY m.name, e.salary DESC
""", "Self-join: employee + their manager")

# Find employees earning more than their manager
run(conn, """
    SELECT e.name AS employee, e.salary, m.name AS manager, m.salary AS mgr_salary
    FROM employees e
    JOIN employees m ON e.manager_id = m.id
    WHERE e.salary > m.salary
""", "Employees earning more than manager")

# Compare consecutive orders: each order paired with previous order for same customer
run(conn, """
    SELECT
        a.customer_id,
        a.order_id AS order_a,
        a.order_date AS date_a,
        a.amount AS amount_a,
        b.order_id AS order_b,
        b.order_date AS date_b,
        b.amount AS amount_b,
        b.amount - a.amount AS delta
    FROM orders a
    JOIN orders b ON a.customer_id = b.customer_id
        AND b.order_date > a.order_date
    WHERE NOT EXISTS (
        SELECT 1 FROM orders c
        WHERE c.customer_id = a.customer_id
          AND c.order_date > a.order_date
          AND c.order_date < b.order_date
    )
    ORDER BY a.customer_id, a.order_date
""", "Self-join: consecutive orders per customer")

## Pattern 3 — Anti-Join

In [ ]:
# Anti-join: rows in A with NO matching row in B
# Three equivalent approaches: LEFT JOIN IS NULL, NOT EXISTS, NOT IN
# Prefer LEFT JOIN IS NULL or NOT EXISTS — NOT IN has NULL behavior bug

# Customers with NO orders
print("=== Anti-join: LEFT JOIN IS NULL ===")
run(conn, """
    SELECT c.customer_id, c.name, c.tier
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.customer_id IS NULL
""", "Customers with no orders (LEFT JOIN IS NULL)")

print("\n=== Anti-join: NOT EXISTS ===")
run(conn, """
    SELECT customer_id, name, tier
    FROM customers c
    WHERE NOT EXISTS (
        SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id
    )
""", "Customers with no orders (NOT EXISTS)")

# Products never ordered
run(conn, """
    SELECT p.product_id, p.name, p.category, p.price
    FROM products p
    LEFT JOIN orders o ON p.product_id = o.product_id
    WHERE o.product_id IS NULL
""", "Products never ordered")

print("""
NOT IN NULL trap:
  SELECT * FROM a WHERE id NOT IN (SELECT id FROM b)
  If b has ANY NULL ids, this returns 0 rows! (NULL comparison is unknown)
  Fix: SELECT * FROM a WHERE id NOT IN (SELECT id FROM b WHERE id IS NOT NULL)
  Or use NOT EXISTS / LEFT JOIN IS NULL instead
""")

## Pattern 4 — CROSS JOIN

In [ ]:
# CROSS JOIN: every row from A paired with every row from B (Cartesian product)
# Rows = |A| × |B|  — use intentionally, never accidentally
# Use cases: generate all combinations, pair every item with every other, fill calendar

# Generate all (region, tier) combinations for a report template
conn2 = sqlite3.connect(":memory:")
conn2.row_factory = sqlite3.Row
conn2.executescript("""
    CREATE TABLE regions(region TEXT);
    INSERT INTO regions VALUES ('East'),('West'),('North'),('South');
    CREATE TABLE tiers(tier TEXT);
    INSERT INTO tiers VALUES ('Gold'),('Silver'),('Bronze');
""")

run(conn2, """
    SELECT region, tier
    FROM regions CROSS JOIN tiers
    ORDER BY region, tier
""", "CROSS JOIN: all region x tier combinations (4x3=12 rows)")

# Practical: fill missing region/tier combos with 0 revenue
run(conn, """
    WITH all_combos AS (
        SELECT DISTINCT c1.region, c2.tier
        FROM customers c1
        CROSS JOIN (SELECT DISTINCT tier FROM customers) c2
    ),
    actuals AS (
        SELECT c.region, c.tier, ROUND(SUM(o.amount), 2) AS revenue
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        WHERE o.status = 'completed'
        GROUP BY c.region, c.tier
    )
    SELECT a.region, a.tier, COALESCE(r.revenue, 0) AS revenue
    FROM all_combos a
    LEFT JOIN actuals r ON a.region = r.region AND a.tier = r.tier
    ORDER BY a.region, a.tier
""", "CROSS JOIN + LEFT JOIN: complete region/tier grid with 0s")

## Pattern 5 — Multi-Table Join Order

In [ ]:
# Multi-table join: order affects intermediate row counts and performance
# Rule: join smallest/most-filtered tables first
# Optimizer may reorder, but explicit CTE pre-filtering is safer

# Three-table join: customers → orders → products
run(conn, """
    SELECT
        c.name AS customer,
        c.tier,
        p.name AS product,
        p.category,
        o.amount,
        o.order_date
    FROM customers c
    INNER JOIN orders o ON c.customer_id = o.customer_id
    INNER JOIN products p ON o.product_id = p.product_id
    WHERE o.status = 'completed'
    ORDER BY c.name, o.order_date
""", "Three-table join: customer + order + product")

# Optimized: pre-filter orders in CTE before joining
print("\nOptimized version with CTE pre-filter:")
run(conn, """
    WITH completed AS (
        SELECT customer_id, product_id, amount, order_date
        FROM orders WHERE status = 'completed'
    )
    SELECT c.name, c.tier, p.name AS product, o.amount, o.order_date
    FROM customers c
    JOIN completed o ON c.customer_id = o.customer_id
    JOIN products p ON o.product_id = p.product_id
    ORDER BY c.name, o.order_date
""", "CTE pre-filter: join on already-filtered orders")

# Aggregated join: avoid fanout when joining to one-to-many
run(conn, """
    -- Wrong: joining then summing can double-count if multiple joins create fanout
    -- Safe: aggregate the many side first, then join
    WITH order_summary AS (
        SELECT customer_id,
               COUNT(*)    AS order_count,
               SUM(amount) AS total_spend
        FROM orders
        WHERE status = 'completed'
        GROUP BY customer_id
    )
    SELECT
        c.name, c.tier, c.region,
        COALESCE(s.order_count, 0) AS orders,
        COALESCE(s.total_spend, 0) AS spend
    FROM customers c
    LEFT JOIN order_summary s ON c.customer_id = s.customer_id
    ORDER BY spend DESC
""", "Aggregate before join: prevent fanout")

## Full Decision Map

```
JOIN TYPE SELECTION
────────────────────
  INNER JOIN          → matched rows only; drops unmatched from both sides
  LEFT JOIN           → all left rows; right cols NULL when no match
  RIGHT JOIN          → all right rows (rarely used; swap to LEFT for clarity)
  FULL OUTER JOIN     → all rows; NULLs where no match (SQLite: UNION of two LEFTs)
  CROSS JOIN          → all combinations; no ON clause
  SELF JOIN           → table aliased twice; compare or traverse within same table

ANTI-JOIN PATTERNS
  LEFT JOIN IS NULL   → preferred (index-friendly, handles NULLs)
  NOT EXISTS          → preferred alternative (correlated but efficient)
  NOT IN              → avoid if subquery can contain NULLs

COMMON BUGS
  Fanout: join to many-side without aggregating first → inflated counts
  NULL join key: NULL = NULL is false → NULLs never match in JOIN condition
  Implicit cross join: SELECT * FROM a, b (old style) = CROSS JOIN
  FULL OUTER not in SQLite → use UNION of LEFT JOINs

PERFORMANCE
  Index join columns on the larger table
  Filter before joining (CTE or subquery)
  Aggregate many-side before joining (avoid fanout + reduces rows)
  Check EXPLAIN for SCAN on join side → add index
```

## Cheat Sheet

```sql
-- Basic joins
SELECT a.*, b.*  FROM a INNER JOIN b ON a.id = b.a_id;    -- matched only
SELECT a.*, b.*  FROM a LEFT  JOIN b ON a.id = b.a_id;    -- all a, null b
SELECT a.*, b.*  FROM a FULL OUTER JOIN b ON a.id=b.a_id; -- all both (not SQLite)
SELECT a.*, b.*  FROM a CROSS JOIN b;                     -- cartesian

-- Anti-join (rows in a with no b match)
SELECT a.* FROM a LEFT JOIN b ON a.id = b.a_id WHERE b.a_id IS NULL;
SELECT a.* FROM a WHERE NOT EXISTS (SELECT 1 FROM b WHERE b.a_id = a.id);

-- Self-join (manager / employee)
SELECT e.name, m.name AS manager
FROM employees e LEFT JOIN employees m ON e.manager_id = m.id;

-- Aggregate before join (prevent fanout)
WITH agg AS (SELECT a_id, SUM(val) total FROM b GROUP BY a_id)
SELECT a.*, COALESCE(agg.total, 0) FROM a LEFT JOIN agg ON a.id = agg.a_id;

-- FULL OUTER JOIN simulation (SQLite)
SELECT a.id, a.name, b.val FROM a LEFT JOIN b ON a.id=b.a_id
UNION
SELECT a.id, a.name, b.val FROM b LEFT JOIN a ON b.a_id=a.id;

-- Multi-table join with filter pushed down
WITH filtered AS (SELECT * FROM orders WHERE status='completed')
SELECT c.name, p.name, f.amount
FROM customers c
JOIN filtered f ON c.id = f.customer_id
JOIN products p ON f.product_id = p.id;

-- Cross join for combination grid
SELECT r.region, t.tier FROM regions r CROSS JOIN tiers t;
```

## Summary Map

```
SQL JOINS — ONE-PAGE SUMMARY
──────────────────────────────

JOIN TYPES
  INNER      → only matching rows
  LEFT       → all left + matched right (NULL for no match)
  RIGHT      → all right + matched left (rarely used)
  FULL OUTER → all both sides (not in SQLite)
  CROSS      → all combinations (explicit intent required)
  SELF       → same table, two aliases

NULL RULES
  NULL = NULL is FALSE in join condition → never matches
  Use IS NULL / IS NOT NULL, not = NULL
  LEFT JOIN + WHERE right.key IS NULL = anti-join

ANTI-JOIN
  LEFT JOIN IS NULL   → preferred
  NOT EXISTS          → preferred alternative
  NOT IN              → avoid (NULL in subquery returns 0 rows silently)

FANOUT
  1-to-many join → correct row count expected
  many-to-many join → unexpected row multiplication
  Fix: aggregate the many side in a CTE before joining

PERFORMANCE
  Index join columns on larger table
  Filter before join (CTE pre-filter)
  Aggregate before join (reduce fanout risk)
  EXPLAIN → SCAN on join side → add index

INTERVIEW SIGNALS
  ✓ Know all join types and when each drops/keeps rows
  ✓ Anti-join: LEFT JOIN IS NULL pattern
  ✓ NOT IN NULL trap (subtle, important)
  ✓ Fanout problem and aggregate-before-join fix
  ✓ Self-join for hierarchy / consecutive row comparison
```